# K-MIMIC-MEDS — ETL Pipeline Validation

This notebook demonstrates and validates the ETL pipeline that converts the **Synthetic K-MIMIC (SYN-ICU)** Korean ICU dataset into the **MEDS (Medical Event Data Standard)** format.

---

## What is MEDS?

MEDS is an international standard for longitudinal medical event data, designed for reproducible machine learning research in healthcare. Every medical observation — a lab result, a vital sign, a diagnosis, a procedure — is represented as a single row with 4 columns:

| Column | Type | Description |
|---|---|---|
| `subject_id` | int64 | Unique patient identifier |
| `time` | timestamp | When the event occurred (`null` = static, e.g. gender) |
| `code` | string | What happened, in hierarchical format (e.g. `CHARTEVENT//HR///min`) |
| `numeric_value` | float32 | Measured value if applicable |

---

## Pipeline Overview

```
data/raw/*.xlsx            (15 source tables, Korean ICU data)
        │
        ▼
   pre_meds.py             (clean IDs, fix timestamps, normalize columns)
        │
        ▼
data/intermediate/*.parquet
        │
        ▼
  meds_convert.py          (extract events, build codes, split patients)
        │
        ▼
data/output/               (MEDS-compliant dataset)
├── data/
│   ├── train/0.parquet
│   ├── tuning/0.parquet
│   └── held_out/0.parquet
└── metadata/
    ├── codes.parquet
    ├── dataset.json
    └── subject_splits.parquet
```

---

## Key Engineering Decisions

| Challenge | Solution |
|---|---|
| K-MIMIC uses UUID string IDs | SHA-256 hashing → stable int64 |
| De-identified years exceed 2262 (pandas ns limit) | Use `datetime64[us]` (microsecond precision) |
| Korean medical units (`회/min`, `℃`, `×10³/㎕`) | `UNIT_MAP` normalization to international standards |
| Diagnoses ICD had no timestamps | Joined with admissions table on `hadm_id` |
| MEDS-Extract CLI incompatible with Windows | Standalone Python pipeline, same output |
| `iterrows()` too slow on 400k+ rows | Vectorized pandas operations (10-50x faster) |

In [66]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/output")

print("Libraries loaded.")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Libraries loaded.
Output directory: C:\Users\33633\Desktop\Efrei\M2\Stage\K-MIMICS-MEDS\data\output


---
## 1. Dataset Metadata

What version of the dataset are we looking at, and when was it produced?

In [67]:
meta = json.loads((OUTPUT_DIR / "metadata/dataset.json").read_text())

print("Dataset metadata:")
print("-" * 40)
for key, value in meta.items():
    print(f"  {key:<20} : {value}")

Dataset metadata:
----------------------------------------
  dataset_name         : K-MIMIC-MEDS
  dataset_version      : 0.2.0
  etl_name             : kmimic-meds
  etl_version          : 0.2.0
  meds_version         : 0.3.3
  created_at           : 2026-04-15T04:26:34.356102+00:00


---
## 2. Output File Structure

Verify that all expected MEDS files are present on disk.

In [68]:
expected = [
    "data/train/0.parquet",
    "data/tuning/0.parquet",
    "data/held_out/0.parquet",
    "metadata/codes.parquet",
    "metadata/dataset.json",
    "metadata/subject_splits.parquet",
]

print("File structure check:")
print("-" * 50)
all_present = True
for f in expected:
    path = OUTPUT_DIR / f
    exists = path.exists()
    size = f"{path.stat().st_size / 1024:.1f} KB" if exists else ""
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {f:<42} {size}")
    if not exists:
        all_present = False

print()
print("✅ All files present." if all_present else "❌ WARNING: some files are missing.")

File structure check:
--------------------------------------------------
  ✅  data/train/0.parquet                       4562.5 KB
  ✅  data/tuning/0.parquet                      638.3 KB
  ✅  data/held_out/0.parquet                    672.8 KB
  ✅  metadata/codes.parquet                     7.1 KB
  ✅  metadata/dataset.json                      0.2 KB
  ✅  metadata/subject_splits.parquet            14.2 KB

✅ All files present.


---
## 3. MEDS Schema Compliance

The MEDS standard requires exactly 4 columns with specific data types.

In [69]:
train = pd.read_parquet(OUTPUT_DIR / "data/train/0.parquet")

print("Column types:")
print("-" * 40)
print(train.dtypes)
print()

schema_checks = {
    "subject_id is int64"      : str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time is datetime[us]"     : "datetime" in str(train["time"].dtype),
    "code is string"           : str(train["code"].dtype) in ("object", "string", "str"),
    "numeric_value is float32" : str(train["numeric_value"].dtype) == "float32",
    "exactly 4 columns"        : len(train.columns) == 4,
}

print("Schema checks:")
print("-" * 40)
for check, result in schema_checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

Column types:
----------------------------------------
subject_id                Int64
time             datetime64[us]
code                        str
numeric_value           float32
dtype: object

Schema checks:
----------------------------------------
  ✅  subject_id is int64
  ✅  time is datetime[us]
  ✅  code is string
  ✅  numeric_value is float32
  ✅  exactly 4 columns


---
## 4. Patient Record Preview

Complete chronological record of one patient.

Static events (no timestamp) come first — gender, then dynamic events in chronological order.

In [70]:
first_patient = train["subject_id"].iloc[0]
patient_df = train[train["subject_id"] == first_patient].reset_index(drop=True)

print(f"Patient {first_patient} — {len(patient_df)} total events")
print()
patient_df.head(20)

Patient 4056806253170987 — 681 total events



,subject_id,time,code,numeric_value
0,4056806253170987,NaT,GENDER//M,NaN
1,4056806253170987,2090-01-01 00:00:01,MEDS_BIRTH,NaN
2,4056806253170987,2130-10-29 11:32:28,ED_REGISTRATION,NaN
3,4056806253170987,2130-10-29 11:53:28,ED_OUT,NaN
4,4056806253170987,2130-10-29 12:38:28,HOSPITAL_ADMISSION//Emergency department,NaN
5,4056806253170987,2130-10-29 12:38:28,INSURANCE//Others,NaN
6,4056806253170987,2130-10-29 12:38:28,MARITAL_STATUS//single,NaN
7,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I251,NaN
8,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I839,NaN
9,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//Others,NaN


In [71]:
print("Event type breakdown for this patient:")
print("-" * 40)
prefixes = patient_df["code"].str.split("//").str[0].value_counts()
for prefix, count in prefixes.items():
    print(f"  {prefix:<25} : {count} events")

Event type breakdown for this patient:
----------------------------------------
  PROCEDURE_START           : 209 events
  PROCEDURE_END             : 209 events
  CHARTEVENT                : 178 events
  LAB                       : 40 events
  OUTPUT                    : 16 events
  INPUT_START               : 12 events
  DIAGNOSIS                 : 3 events
  GENDER                    : 1 events
  MEDS_BIRTH                : 1 events
  ED_REGISTRATION           : 1 events
  ED_OUT                    : 1 events
  HOSPITAL_ADMISSION        : 1 events
  INSURANCE                 : 1 events
  MARITAL_STATUS            : 1 events
  ICU_ADMISSION             : 1 events
  ICU_DISCHARGE             : 1 events
  PROCEDURE_ICD             : 1 events
  HOSPITAL_DISCHARGE        : 1 events
  MEDICATION                : 1 events


---
## 5. Dataset Statistics

Global numbers across all splits.

In [72]:
all_dfs = []
for split_name in ["train", "tuning", "held_out"]:
    df = pd.read_parquet(OUTPUT_DIR / f"data/{split_name}/0.parquet")
    df["split"] = split_name
    all_dfs.append(df)

full = pd.concat(all_dfs, ignore_index=True)

print("=" * 45)
print("GLOBAL STATISTICS")
print("=" * 45)
print(f"  Total events          : {len(full):>12,}")
print(f"  Total patients        : {full['subject_id'].nunique():>12,}")
print(f"  Static events (NaT)   : {full['time'].isna().sum():>12,}")
print(f"  Dynamic events        : {full['time'].notna().sum():>12,}")
print(f"  With numeric value    : {full['numeric_value'].notna().sum():>12,}")
print(f"  Unique codes          : {full['code'].nunique():>12,}")
print()
print("Events and patients per split:")
print("-" * 45)
summary = full.groupby("split").agg(
    events=("code", "count"),
    patients=("subject_id", "nunique")
)
print(summary)

GLOBAL STATISTICS
  Total events          :    1,381,668
  Total patients        :        1,328
  Static events (NaT)   :        1,328
  Dynamic events        :    1,380,340
  With numeric value    :      605,941
  Unique codes          :          201

Events and patients per split:
---------------------------------------------
           events  patients
split                      
held_out   145664       134
train     1090801      1062
tuning     135790       132


---
## 6. Event Type Distribution

How many events come from each source table?

In [73]:
full["event_type"] = full["code"].str.split("//").str[0]

print("Event distribution by type:")
print("-" * 55)
dist = full["event_type"].value_counts()
for event_type, count in dist.items():
    pct = count / len(full) * 100
    bar = "█" * int(pct / 2)
    print(f"  {event_type:<25} : {count:>9,}  ({pct:4.1f}%)  {bar}")

Event distribution by type:
-------------------------------------------------------
  CHARTEVENT                :   416,645  (30.2%)  ███████████████
  PROCEDURE_START           :   371,107  (26.9%)  █████████████
  PROCEDURE_END             :   371,107  (26.9%)  █████████████
  LAB                       :   128,249  ( 9.3%)  ████
  OUTPUT                    :    36,353  ( 2.6%)  █
  INPUT_START               :    26,691  ( 1.9%)  
  MEDICATION                :     5,675  ( 0.4%)  
  DIAGNOSIS                 :     3,091  ( 0.2%)  
  PROCEDURE_ICD             :     1,479  ( 0.1%)  
  GENDER                    :     1,328  ( 0.1%)  
  HOSPITAL_ADMISSION        :     1,328  ( 0.1%)  
  MARITAL_STATUS            :     1,328  ( 0.1%)  
  ICU_ADMISSION             :     1,328  ( 0.1%)  
  ICU_DISCHARGE             :     1,328  ( 0.1%)  
  HOSPITAL_DISCHARGE        :     1,328  ( 0.1%)  
  INSURANCE                 :     1,327  ( 0.1%)  
  MEDS_BIRTH                :     1,249  ( 0.1%)  
  E

---
## 7. Patient Split Distribution

Patients are randomly split 80/10/10 with a fixed seed for reproducibility.

In [74]:
splits = pd.read_parquet(OUTPUT_DIR / "metadata/subject_splits.parquet")
total = len(splits)
counts = splits["split"].value_counts()

print("Patient split distribution:")
print("-" * 40)
for split_name in ["train", "tuning", "held_out"]:
    count = counts.get(split_name, 0)
    pct = count / total * 100
    print(f"  {split_name:<12} : {count:>5} patients ({pct:.1f}%)")

print()
split_checks = {
    "train ~80%"    : 75 <= counts.get("train", 0) / total * 100 <= 85,
    "tuning ~10%"   : 8  <= counts.get("tuning", 0) / total * 100 <= 12,
    "held_out ~10%" : 8  <= counts.get("held_out", 0) / total * 100 <= 12,
}
for check, result in split_checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

Patient split distribution:
----------------------------------------
  train        :  1062 patients (80.0%)
  tuning       :   132 patients (9.9%)
  held_out     :   134 patients (10.1%)

  ✅  train ~80%
  ✅  tuning ~10%
  ✅  held_out ~10%


---
## 8. Code Catalog

Every unique code in the dataset is documented in `codes.parquet`, enriched with descriptions from K-MIMIC item dictionaries and linked to the EDI (Korean health insurance) ontology for lab codes.

In [75]:
codes = pd.read_parquet(OUTPUT_DIR / "metadata/codes.parquet")

print(f"Total unique codes          : {len(codes)}")
print(f"With description            : {codes['description'].notna().sum()}")
print(f"With parent_codes (EDI)     : {codes['parent_codes'].notna().sum()}")
print()
print("Sample — LAB codes with EDI parent codes:")
codes[codes["parent_codes"].notna()][["code", "description", "parent_codes"]].head(8)

Total unique codes          : 201
With description            : 110
With parent_codes (EDI)     : 32

Sample — LAB codes with EDI parent codes:


,code,description,parent_codes
80,LAB//001L2001//x10e3/uL,WBC(검사24시간가능),[EDI/D0002014]
81,LAB//001L2002//x10e6/uL,RBC(검사24시간가능),[EDI/D0002034]
82,LAB//001L2003//g/dL,Hb(검사24시간가능),[EDI/D0002054]
83,LAB//001L2004//%,Hct,[EDI/D0002044]
84,LAB//001L2005//%,RDW(검사24시간가능),[EDI/D0002024]
85,LAB//001L2006//fL,PDW(검사24시간가능),[EDI/D0002064]
86,LAB//001L2009//x10e3/uL,PLT(검사24시간가능),[EDI/D0002074]
102,LAB//001L2204//sec,aPTT (검사24시간가능),[EDI/D1004004]


---
## 9. Diagnosis Timestamps

Diagnoses are linked to their admission timestamp via a join on `hadm_id`, making them dynamic events rather than static ones.

In [76]:
diag = full[full["code"].str.startswith("DIAGNOSIS")]

print(f"Total diagnoses            : {len(diag):,}")
print(f"With timestamp             : {diag['time'].notna().sum():,}")
print(f"Without timestamp (NaT)    : {diag['time'].isna().sum():,}")
print()
print("Sample diagnosis events:")
diag[["subject_id", "time", "code"]].head(8)

Total diagnoses            : 3,091
With timestamp             : 3,091
Without timestamp (NaT)    : 0

Sample diagnosis events:


,subject_id,time,code
7,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I251
8,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I839
9,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//Others
686,14397825242364671,2846-06-20 02:56:00,DIAGNOSIS//KCD8//Others
1867,21501110057397942,2387-11-01 00:00:18,DIAGNOSIS//KCD8//I509
1868,21501110057397942,2387-11-01 00:00:18,DIAGNOSIS//KCD8//Others
2611,42028212112143695,2319-08-09 20:49:29,DIAGNOSIS//KCD8//N179
2612,42028212112143695,2319-08-09 20:49:29,DIAGNOSIS//KCD8//Others


---
## 10. Unit Normalization

K-MIMIC uses Korean and non-standard units. The pipeline normalizes them to international standards.

In [77]:
korean = full[full["code"].str.contains("회|℃|㎍|㎎|×|㎕|μℓ", na=False, regex=True)]
print(f"Codes with non-standard units : {len(korean)}")
print(f"Result: {'✅ All units normalized' if len(korean) == 0 else '❌ Some units not normalized'}")
print()
print("Standard units present in codes (top 10):")
print("-" * 40)
units = (
    full["code"]
    .str.extract(r"//([^/]+)$")[0]
    .dropna()
    .value_counts()
)
unit_only = units[~units.index.str.match(r"^[0-9A-Z]{3,}_")].head(10)
for unit, count in unit_only.items():
    print(f"  {unit:<15} : {count:,}")

Codes with non-standard units : 0
Result: ✅ All units normalized

Standard units present in codes (top 10):
----------------------------------------
  mmHg            : 187,235
  min             : 67,330
  cc              : 63,044
  %               : 52,771
  cmH2O           : 26,513
  Cel             : 26,467
  ml              : 16,585
  fL              : 7,791
  sec             : 3,996
  uL              : 3,650


---
## 11. ICU Procedure Events

ICU procedures (ventilation, catheters, monitoring) extracted with start and end timestamps.

In [78]:
proc_start = full[full["code"].str.startswith("PROCEDURE_START")]
proc_end   = full[full["code"].str.startswith("PROCEDURE_END")]

print(f"PROCEDURE_START events     : {len(proc_start):,}")
print(f"PROCEDURE_END events       : {len(proc_end):,}")
print(f"All have timestamps        : {'✅' if proc_start['time'].notna().all() else '❌'}")
print()
print("Most frequent ICU procedures:")
print("-" * 50)
for code, count in proc_start["code"].value_counts().head(5).items():
    print(f"  {code:<40} : {count:,}")

PROCEDURE_START events     : 371,107
PROCEDURE_END events       : 371,107
All have timestamps        : ✅

Most frequent ICU procedures:
--------------------------------------------------
  PROCEDURE_START//001P_OPFG130303         : 33,737
  PROCEDURE_START//001P_OPFG140904         : 33,737
  PROCEDURE_START//001P_L9881              : 33,737
  PROCEDURE_START//001P_RG3011             : 33,737
  PROCEDURE_START//001P_KL1442             : 33,737


---
## 12. Full Validation Summary

In [79]:
print("=" * 50)
print("VALIDATION SUMMARY")
print("=" * 50)
print()

all_checks = {
    "All output files present"       : all((OUTPUT_DIR / f).exists() for f in expected),
    "subject_id is int64"            : str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time is datetime"               : "datetime" in str(train["time"].dtype),
    "code is string"                 : str(train["code"].dtype) in ("object", "string", "str"),
    "numeric_value is float32"       : str(train["numeric_value"].dtype) == "float32",
    "No //nan in codes"              : len(train[train["code"].str.contains("//nan", na=False)]) == 0,
    "No UNKNOWN codes"               : len(train[train["code"] == "UNKNOWN"]) == 0,
    "No Korean units"                : len(full[full["code"].str.contains("회|℃|㎍|㎎|×|㎕", na=False, regex=True)]) == 0,
    "Static events are GENDER only"  : full[full["time"].isna()]["code"].str.startswith("GENDER").all(),
    "Diagnoses have timestamps"      : full[full["code"].str.startswith("DIAGNOSIS")]["time"].notna().all(),
    "Procedure events present"       : len(full[full["code"].str.startswith("PROCEDURE_START")]) > 0,
    "Splits 80/10/10"                : all(split_checks.values()),
    "codes.parquet has descriptions" : codes["description"].notna().sum() > 0,
    "codes.parquet has parent_codes" : codes["parent_codes"].notna().sum() > 0,
    "dataset.json is present"        : (OUTPUT_DIR / "metadata/dataset.json").exists(),
}

for check, result in all_checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

passed = sum(all_checks.values())
total  = len(all_checks)
print()
print("=" * 50)
print(f"  Result: {passed}/{total} checks passed")
print("=" * 50)

VALIDATION SUMMARY

  ✅  All output files present
  ✅  subject_id is int64
  ✅  time is datetime
  ✅  code is string
  ✅  numeric_value is float32
  ✅  No //nan in codes
  ✅  No UNKNOWN codes
  ✅  No Korean units
  ✅  Static events are GENDER only
  ✅  Diagnoses have timestamps
  ✅  Procedure events present
  ✅  Splits 80/10/10
  ✅  codes.parquet has descriptions
  ✅  codes.parquet has parent_codes
  ✅  dataset.json is present

  Result: 15/15 checks passed
